In [1]:
import torch
import torch.multiprocessing as mp
from gymnasium.vector import AsyncVectorEnv
import numpy as np
from env import mbb
from models import ppo
import os
import pandas as pd
import matplotlib.pyplot as plt
import logging
from datetime import datetime
import random
from torch import amp
from env import mbb_cnn
from models import ppo_cnn


In [2]:

# Configure logging
logging.basicConfig(
    level=logging.INFO,  # Set to DEBUG for more detailed logs
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("reward_logs.log"),  # Log to a file
        logging.StreamHandler()  # Also log to console
    ]
)


print("============================================================================================")
device = torch.device('cpu')
if(torch.cuda.is_available()): 
    device = torch.device('cuda:0') 
    torch.cuda.empty_cache()
    print("Device set to : " + str(torch.cuda.get_device_name(device)))
else:
    print("Device set to : cpu")  
print("============================================================================================")


Device set to : NVIDIA GeForce RTX 4070 Laptop GPU


In [3]:
def make_beam_env():
    """
    Factory function to create a single instance of BeamOptimizationEnv.
    Randomly selects a beam_type from [1,5].
    """
    beam_type = random.choice([1, 5])
    return mbb_cnn.BeamOptimizationEnv(width=6, height=6, density=0.4, optimal_density=0.5, reward_weights=None, beam_type=beam_type)

if __name__ == "__main__":
    mp.set_start_method('spawn', force=True)

    # Set random seeds for reproducibility
    random_seed = 0
    torch.manual_seed(random_seed)
    np.random.seed(random_seed)
    random.seed(random_seed)

    # Hyperparameters
    env_name = "BeamOptimizationEnv"
    num_envs = 32
    width = 6
    height = 6
    channels = 5  # Densities, Normals X & Y, Forces X & Y
    action_dim = 2 * width * height  # Assuming discrete actions: increase or decrease each cell
    has_continuous_action_space = False
    max_ep_len = width * height * 2
    max_training_timesteps = int(5e5)
    update_timestep = max_ep_len  # Update PPO every 'update_timestep' timesteps

    # Action std decay parameters (only for continuous action spaces)
    action_std_decay_freq = 5000
    action_std_decay_rate = 0.05
    min_action_std = 0.1

    # Define the environment functions for vectorized environments
    env_fns = [make_beam_env for _ in range(num_envs)]  # Pass the function itself

    # Create vectorized environments
    vector_env = AsyncVectorEnv(env_fns)
    print(f"Vector Environment Type: {type(vector_env)}")

    # Get state and action dimensions
    state_shape = vector_env.single_observation_space.shape  # Should be (channels, width+1, height+1)
    print(f"State shape: {state_shape}")

    state_channels = state_shape[0]
    state_width = state_shape[1]
    state_height = state_shape[2]

    if has_continuous_action_space:
        action_dim = vector_env.single_action_space.shape[0]
    else:
        action_dim = vector_env.single_action_space.n

    print(f"Action dimension: {action_dim}")

    # Initialize PPO agent
    ppo_agent = ppo_cnn.PPO(
        state_channels=state_channels,
        width=width,
        height=height,
        action_dim=action_dim,
        lr_actor=0.0003,
        lr_critic=0.001,
        gamma=0.95,
        K_epochs=3,
        eps_clip=0.3,
        has_continuous_action_space=has_continuous_action_space,
        action_std_init=0.6,
        gae_lambda=0.95,
        num_envs=num_envs,
        device=device
    )

    # Track total training time
    start_time = datetime.now().replace(microsecond=0)
    print("Started training at (GMT) :", start_time)
    print("============================================================================================")

    ###################### Logging ######################
    # Directory setup
    log_dir = "PPO_logs"
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)

    log_dir = os.path.join(log_dir, env_name)
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)

    # Determine run number based on existing log files
    run_num = len([name for name in os.listdir(log_dir) if os.path.isfile(os.path.join(log_dir, name))])
    log_f_name = os.path.join(log_dir, f'PPO_{env_name}_log_{run_num}.csv')

    print(f"Current logging run number for {env_name}: {run_num}")
    print(f"Logging at: {log_f_name}")

    #####################################################

    ################### Checkpointing ###################
    run_num_pretrained = 0  # Change this to prevent overwriting weights in the same env_name folder

    checkpoint_dir = "PPO_preTrained"
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_dir = os.path.join(checkpoint_dir, env_name)
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)

    checkpoint_path = os.path.join(checkpoint_dir, f"PPO_{env_name}_{random_seed}_{run_num_pretrained}.pth")
    print(f"Save checkpoint path: {checkpoint_path}")

    #####################################################

    # Initialize logging file
    with open(log_f_name, "w+") as log_f:
        log_f.write('episode,timestep,average_reward\n')

    print_running_reward = 0
    log_running_reward = 0
    print_running_episodes = 0
    log_running_episodes = 0

    # Initialize training variables
    timestep = 0
    i_episode = 0

    #################### Training Loop ####################
    while timestep < max_training_timesteps:
        # Reset all environments and unpack the observations and info
        states, infos = vector_env.reset()
        # Convert states to torch tensors
        states = torch.tensor(states, dtype=torch.float32).to(device)  # Shape: [num_envs, channels, W, H]
        ep_rewards = np.zeros(num_envs)
        done_flags = np.array([False] * num_envs)

        for t_step in range(1, max_ep_len + 1):
            # Select actions for all environments
            actions, logprobs, state_values = ppo_agent.select_action(states)

            # Take actions in all environments
            try:
                next_states, rewards, dones, truncateds, infos = vector_env.step(actions)
            except Exception as e:
                print(f"Error during environment step: {e}")
                break

            # Combine done flags
            dones_combined = np.logical_or(dones, truncateds)

            # Convert next_states to torch tensors
            next_states = torch.tensor(next_states, dtype=torch.float32).to(device)

            # Convert actions to tensors
            if not ppo_agent.has_continuous_action_space:
                actions_tensor = torch.LongTensor(actions).to(device)  # [num_envs]
            else:
                actions_tensor = torch.FloatTensor(actions).to(device)  # [num_envs]

            # Store transitions in the buffer
            ppo_agent.buffer.store(
                states=states,  # [num_envs, channels, W, H]
                actions=actions_tensor,  # [num_envs]
                logprobs=torch.tensor(logprobs, dtype=torch.float32).to(device),  # [num_envs]
                rewards=torch.tensor(rewards, dtype=torch.float32).to(device),  # [num_envs]
                state_values=torch.tensor(state_values, dtype=torch.float32).to(device),  # [num_envs]
                is_terminals=torch.tensor(dones_combined, dtype=torch.bool).to(device)  # [num_envs]
            )

            ep_rewards += rewards
            states = next_states
            timestep += 1

            # Update PPO agent if enough timesteps have passed
            if timestep % update_timestep == 0:
                ppo_agent.update()

            # For continuous action spaces, decay action std
            if ppo_agent.has_continuous_action_space and timestep % action_std_decay_freq == 0:
                ppo_agent.decay_action_std(action_std_decay_rate, min_action_std)

            # Log in logging file
            if timestep % update_timestep == 0:
                # Compute average reward
                if log_running_episodes > 0:
                    log_avg_reward = log_running_reward / log_running_episodes
                else:
                    log_avg_reward = 0
                log_avg_reward = np.mean(log_avg_reward)
                log_avg_reward = round(log_avg_reward, 4)

                with open(log_f_name, "a") as log_f:
                    log_f.write(f'{i_episode},{timestep},{log_avg_reward}\n')

                log_running_reward = 0
                log_running_episodes = 0

            # Print average reward
            if timestep % (update_timestep * 2) == 0:
                # Compute average reward
                if print_running_episodes > 0:
                    print_avg_reward = print_running_reward / print_running_episodes
                else:
                    print_avg_reward = 0
                print_avg_reward = np.mean(print_avg_reward)
                print_avg_reward = round(print_avg_reward, 2)

                print(f"Episode : {i_episode} \t\t Timestep : {timestep} \t\t Average Reward : {print_avg_reward}")

                print_running_reward = 0
                print_running_episodes = 0

            # Save model weights
            if timestep % int(1e3) == 0:
                print("--------------------------------------------------------------------------------------------")
                print(f"Saving model at {checkpoint_path}")
                ppo_agent.save(checkpoint_path)
                print("Model saved.")
                elapsed_time = datetime.now().replace(microsecond=0) - start_time
                print(f"Elapsed Time: {elapsed_time}")
                print("--------------------------------------------------------------------------------------------")

            # Break if all environments are done
            if dones_combined.all():
                break

        # Logging for the episode
        print_running_reward += ep_rewards.sum()
        print_running_episodes += 1

        log_running_reward += ep_rewards.sum()
        log_running_episodes += 1

        i_episode += 1

    # Close environments after training
    vector_env.close()

    # Print total training time
    end_time = datetime.now().replace(microsecond=0)
    print("============================================================================================")
    print("Started training at (GMT) :", start_time)
    print("Finished training at (GMT) :", end_time)
    print(f"Total training time : {end_time - start_time}")
    print("============================================================================================")

    # Plot training metrics
    ppo_agent.plot_metrics()

Vector Environment Type: <class 'gymnasium.vector.async_vector_env.AsyncVectorEnv'>
State shape: (5, 7, 7)
Action dimension: 72
Started training at (GMT) : 2025-01-12 12:00:24
Current logging run number for BeamOptimizationEnv: 1
Logging at: PPO_logs/BeamOptimizationEnv/PPO_BeamOptimizationEnv_log_1.csv
Save checkpoint path: PPO_preTrained/BeamOptimizationEnv/PPO_BeamOptimizationEnv_0_0.pth


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x6272 and 4608x256)